# Question Pipeline on Google Colab

This notebook contains the question_pipeline source code. It downloads every PDF from the Google Drive folder ID you provide, verifies downloads, runs chunked MinerU extraction with resumable checkpoints, handles chart visuals, and optionally structures questions with OpenRouter into Excel. You do not need to clone GitHub or upload the source project separately.

Mount your Drive, paste the PDF folder ID in the setup cell, then run the cells in order. PDF inputs, checkpoints, and outputs persist in `MyDrive/QuestionPipelineColab`; models use Colab runtime storage. Select a GPU runtime before extraction.

OpenRouter is optional. Questions remain marked for human review.

In [ ]:
from google.colab import drive
from pathlib import Path
import base64, io, os, shutil, subprocess, sys, zipfile

drive.mount('/content/drive')

# Embedded project source: no GitHub clone or separate source-file upload is needed.
PIPELINE_BUNDLE_B64 = (
    '...'
)
SOURCE_ROOT = Path('/content/question_pipeline_source')
if SOURCE_ROOT.exists():
    shutil.rmtree(SOURCE_ROOT)
SOURCE_ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(base64.b64decode(PIPELINE_BUNDLE_B64))) as source_zip:
    source_zip.extractall(SOURCE_ROOT)

# Paste the ID from the Google Drive folder URL that contains the PDFs.
DRIVE_FOLDER_ID = 'PASTE_GOOGLE_DRIVE_FOLDER_ID_HERE'
PERSISTENT = Path('/content/drive/MyDrive/QuestionPipelineColab')
RUNTIME_ROOT = Path('/content/question_pipeline_colab')
PACKAGE_SOURCE = SOURCE_ROOT / 'question_pipeline'
PACKAGE = RUNTIME_ROOT / 'question_pipeline'

assert (PACKAGE_SOURCE / 'run.py').is_file(), f'Embedded run.py not found: {PACKAGE_SOURCE}'
assert DRIVE_FOLDER_ID != 'PASTE_GOOGLE_DRIVE_FOLDER_ID_HERE', 'Paste the Drive folder ID first.'
print('Embedded pipeline source:', PACKAGE_SOURCE)
print('Drive folder ID:', DRIVE_FOLDER_ID)


## Prepare a Colab workspace

The code is copied into Colab's local disk for faster file access. Only `data` and `outputs` are linked to the persistent Google Drive folder. Your local credentials and unrelated pipelines are not copied.

In [ ]:
if RUNTIME_ROOT.exists():
    shutil.rmtree(RUNTIME_ROOT)
RUNTIME_ROOT.mkdir(parents=True)
shutil.copytree(PACKAGE_SOURCE, PACKAGE, ignore=shutil.ignore_patterns('data', 'outputs', 'secrets', '__pycache__'))

PERSISTENT.mkdir(parents=True, exist_ok=True)
for name in ('data', 'outputs'):
    target = PERSISTENT / name
    target.mkdir(parents=True, exist_ok=True)
    (PACKAGE / name).symlink_to(target, target_is_directory=True)

# Isolate MinerU/Transformers downloads to this Colab session.
os.environ['MINERU_HOME'] = '/content/mineru-model-cache'
os.environ['HF_HOME'] = '/content/huggingface-cache'
os.environ['MINERU_PROCESSING_WINDOW_SIZE'] = '2'
print('Runtime code:', PACKAGE)
print('Persistent state:', PERSISTENT)

## Install the extraction and chart dependencies

This installs the versions declared by the project. MinerU, DePlot and their model weights can take time and disk space to download. Restarting a cell or Colab session does not erase completed extraction checkpoints on Drive.

In [ ]:
subprocess.run([sys.executable, '-m', 'pip', 'install', '-r', str(PACKAGE / 'requirements.txt')], check=True)

# Colab's system Python may not include ensurepip, so use virtualenv to
# create the chart worker environment while reusing Colab's GPU-enabled PyTorch.
subprocess.run([sys.executable, '-m', 'pip', 'install', 'virtualenv'], check=True)
CHART_ENV = RUNTIME_ROOT / '.venv-charts'
CHART_PYTHON = CHART_ENV / 'bin' / 'python'
if not CHART_PYTHON.exists():
    if CHART_ENV.exists():
        shutil.rmtree(CHART_ENV)
    subprocess.run([sys.executable, '-m', 'virtualenv', '--system-site-packages', str(CHART_ENV)], check=True)
subprocess.run([str(CHART_PYTHON), '-m', 'pip', 'install', '-r', str(PACKAGE / 'requirements-charts.txt')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', 'openpyxl>=3.1,<4', 'google-api-python-client>=2,<3', 'google-auth>=2,<3'], check=True)
print('Dependencies installed. Select Runtime → Change runtime type → GPU if you have not already.')

## Download every PDF from the Drive folder

The folder ID is the part after `/folders/` in the Google Drive URL. Colab will ask you to authorize Drive access. This uses the pipeline's paginated listing and resumable, checksum-verified downloader. It downloads every PDF directly inside the selected folder, and this cell must finish before OCR starts.

In [ ]:
from google.colab import auth
import google.auth
from google.auth.transport.requests import AuthorizedSession
from googleapiclient.discovery import build
sys.path.insert(0, str(PACKAGE))
from ingestion.drive import Drive

auth.authenticate_user()
credentials, _ = google.auth.default()
# Use the existing downloader with Colab credentials instead of local OAuth files.
downloader = object.__new__(Drive)
downloader.service = build('drive', 'v3', credentials=credentials, cache_discovery=False)
downloader.session = AuthorizedSession(credentials)
try:
    remote_pdfs = downloader.list_pdfs(DRIVE_FOLDER_ID)
    if not remote_pdfs:
        raise RuntimeError('No PDF files found directly inside this Drive folder.')
    print(f'Found {len(remote_pdfs)} PDF(s). Downloading all before OCR:')
    for index, item in enumerate(remote_pdfs, 1):
        print(f'[{index}/{len(remote_pdfs)}] {item["name"]}', flush=True)
        downloader.download(item)
finally:
    downloader.close()

PDF_FOLDER = PACKAGE / 'data' / 'downloads'
pdfs = sorted(path for path in PDF_FOLDER.iterdir() if path.is_file() and path.suffix.lower() == '.pdf')
print(f'All {len(pdfs)} PDF(s) are downloaded and verified. Ready for OCR.')

## Extract PDFs and create Markdown

The pipeline processes PDFs sequentially in five-page windows with one overlap page. Completed extraction chunks, chart analyses and Markdown pages are checkpointed. Run the same cell again after an interruption to resume. This cell stops after Markdown; it does not contact OpenRouter or create a workbook.

In [ ]:
# Optional quick test: set MAX_PAGES = 2. Use None for complete PDFs.
MAX_PAGES = None
VERIFY_CHART_TEXT = False  # PaddleOCR cross-check is optional and is not installed here.

RUN_SCRIPT = PACKAGE / 'run.py'
PDF_FOLDER = PACKAGE / 'data' / 'downloads'
if not RUN_SCRIPT.is_file():
    raise FileNotFoundError(f'Pipeline run.py is missing: {RUN_SCRIPT}')
pdfs = sorted(path for path in PDF_FOLDER.iterdir() if path.is_file() and path.suffix.lower() == '.pdf')
if not pdfs:
    raise FileNotFoundError(f'No downloaded PDFs found in {PDF_FOLDER}. Run the Drive download cell first.')
print(f'Starting extraction for {len(pdfs)} PDF(s) from {PDF_FOLDER}.', flush=True)

command = [sys.executable, str(RUN_SCRIPT), str(PDF_FOLDER), '--extract-only']
if MAX_PAGES is not None:
    command += ['--max-pages', str(MAX_PAGES)]
if VERIFY_CHART_TEXT:
    command.append('--verify-chart-text')
subprocess.run(command, cwd=PACKAGE, check=True)
print('Extraction stage finished. Markdown and checkpoints are under:', PACKAGE / 'outputs')


## Optional: structure questions with OpenRouter and export Excel

Enter a model that supports strict function/tool calling. The API key is prompted privately and kept in the current Colab process. Each validated batch is cached for resume. If the credentials or model are blank, no request is made. The workbook is generated with OpenPyXL and saved beside the structured bank on Drive.

In [ ]:
from getpass import getpass
import hashlib, json
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment
from openpyxl.utils import get_column_letter

model = input('OpenRouter model ID (blank skips this stage): ').strip()
api_key = getpass('OpenRouter API key (input is hidden): ').strip() if model else ''
if not model or not api_key:
    print('Skipped OpenRouter and Excel. Extracted Markdown is ready under:', PACKAGE / 'outputs')
else:
    os.environ['OPENROUTER_API_KEY'] = api_key
    sys.path.insert(0, str(PACKAGE))
    from core.storage import read_json, sha256, write_json
    from ai.structure import structure

    config = read_json(PACKAGE / 'config.json')
    config['openrouter_model'] = model
    config['openrouter_enabled'] = True
    output_root = PACKAGE / 'outputs'
    manifests = sorted(output_root.rglob('manifest.json'))
    processed = []
    for manifest_path in manifests:
        state = read_json(manifest_path, {})
        if state.get('status') not in ('complete', 'partial'):
            continue
        markdown = manifest_path.with_name('markdown.md')
        if not markdown.is_file():
            continue
        bank_path = structure(markdown, state['source_sha256'], config)
        if bank_path is None:
            continue
        bank = read_json(bank_path)
        source_text = markdown.read_text(encoding='utf-8')
        content_hash = sha256(bank_path)
        workbook_path = bank_path.with_name(f'questions-{content_hash[:12]}.xlsx')
        if workbook_path.exists():
            print('Preserving existing workbook:', workbook_path)
            processed.append(workbook_path)
            continue

        workbook = Workbook()
        questions_sheet = workbook.active
        questions_sheet.title = 'Questions'
        headers = ['question_number','question_text','options','correct_answer','explanation','source_page','question_type','review_status','review_notes','diagram_markdown','source_quote','question_id']
        questions_sheet.append(headers)
        for row in bank.get('questions', []):
            options = '\n'.join(f"{item.get('label', '')}. {item.get('text', '')}" for item in row.get('options', []))
            questions_sheet.append([row.get('question_number',''), row.get('question_text',''), options, row.get('correct_answer',''), row.get('explanation',''), row.get('source_page',''), row.get('question_type',''), 'PENDING', row.get('review_notes',''), row.get('diagram_markdown',''), row.get('source_quote',''), row.get('question_id','')])
        source_sheet = workbook.create_sheet('Source')
        source_sheet.append(['part', 'markdown'])
        for part, start in enumerate(range(0, len(source_text), 30000), 1):
            source_sheet.append([part, source_text[start:start+30000]])
        review_sheet = workbook.create_sheet('Review')
        review_sheet.append(['source_page', 'unresolved_content'])
        for item in bank.get('unresolved', []):
            review_sheet.append([item.get('source_page',''), item.get('text','')])
        for sheet in workbook.worksheets:
            sheet.freeze_panes = 'A2'
            sheet.row_dimensions[1].height = 28
            for cell in sheet[1]:
                cell.font = Font(bold=True, color='FFFFFF')
                cell.fill = PatternFill('solid', fgColor='2E7D32')
                cell.alignment = Alignment(wrap_text=True, vertical='top')
            for column in sheet.columns:
                letter = get_column_letter(column[0].column)
                sheet.column_dimensions[letter].width = min(48, max(14, max(len(str(cell.value or '')) for cell in column[:50]) + 2))
                for cell in column[1:]:
                    cell.alignment = Alignment(wrap_text=True, vertical='top')
                    cell.data_type = 's'  # Keep source text literal; do not turn it into a formula.
        workbook.save(workbook_path)
        processed.append(workbook_path)
        print('Excel ready for human review:', workbook_path)
    print(f'Finished {len(processed)} workbook(s). Questions are still pending human review.')

## Resume and inspect results

If the runtime disconnects, reconnect Google Drive, rerun setup cells, and rerun the extraction cell. Saved extraction/chart/page checkpoints are reused. To retry only OpenRouter and workbook export, rerun the last cell; successful OpenRouter batches are cached.

Find outputs in `MyDrive/QuestionPipelineColab/outputs`. Each book contains the merged `markdown.md`, source-page files, structured `bank.json`, and a content-versioned Excel workbook. Review questions and chart reconstructions against the original PDF before training or publishing. This notebook does not upload data to Supabase.